# Does the tilt preference follow the direction of background shear?

**Primary question:** do AEs tilt left and CEs right across different geographic shear directions, separately in planetary and topographic eddy-days?

We also test whether topographic eddies encounter more variable shear. This is a hypothesis, not a selection assumption. No individual case studies are selected.

The mixed population and **all quality-controlled eddy-days**, including unknown PV labels, test generalisation directly. These populations overlap; their estimates are not independent. “All” does not include shallow columns, weak shear or poorly defined tilt. Evidence in both dominant regimes alone does not establish the result for every eddy.

Run after `00_prepare_inputs.ipynb`; notebook 01 need not be rerun. No new model-field extraction. All direction conventions and geometry remain those of notebook 01.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p/'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None: raise FileNotFoundError('Launch from the analysis folder or this subfolder.')
THIS_ROOT = ANALYSIS_ROOT/'shear_tilt_climatology'
sys.path.insert(0, str(THIS_ROOT))
import climatology_tools as ct
import direction_tools as dt
CACHE_ROOT = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/shear_tilt_climatology')
INPUT_PATH = CACHE_ROOT/'climatology_inputs.parquet'
FAMILY = 'clim'
DEFINITION = 'upper_deep'
MIN_TILT_KM = 5.0
MIN_SHEAR_MS = 0.005
BOOTSTRAPS = 500
MIN_EDDIES = 20
MIN_ROWS = 50
SEED = 20260916
WINDOW_DAYS = 14
TURN_LAGS = (3, 7, 14)
PRIMARY_TURN_LAG = 7
RUN_SENSITIVITIES = True
SAVE_OUTPUTS = True
raw = pd.read_parquet(INPUT_PATH)
ct.validate_days(raw)
def prepare(**overrides):
    opts = dict(family=FAMILY, definition=DEFINITION, min_tilt=MIN_TILT_KM, min_shear=MIN_SHEAR_MS)
    opts.update({k:v for k,v in overrides.items() if k in opts})
    labels = ct.classify_regimes(raw, factor=overrides.get('factor',2.), window=7, min_periods=5,
                                planetary_depth=3000., smoothing='trailing')
    return ct.relative_geometry(labels, **opts)
data = prepare()
pop = dt.populations(data)
if pop.empty: raise ValueError('No usable geometry: inspect the cached inputs.')
OUTPUT = CACHE_ROOT/'direction_results'/pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
settings = dict(family=FAMILY, definition=DEFINITION, min_tilt_km=MIN_TILT_KM,
    min_shear_ms=MIN_SHEAR_MS, bootstraps=BOOTSTRAPS, min_eddies=MIN_EDDIES, min_rows=MIN_ROWS,
    seed=SEED, window_days=WINDOW_DAYS, turn_lags=TURN_LAGS, primary_turn_lag=PRIMARY_TURN_LAG,
    dominance_factor=2, regime_window=7, regime_min_periods=5, planetary_depth=3000,
    sector_width_degrees=45, sensitivity_run=RUN_SENSITIVITIES)
if SAVE_OUTPUTS:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    provenance = dict(settings=settings,
        input_metadata=json.loads((CACHE_ROOT/'input_metadata.json').read_text()),
        input_file=dict(path=str(INPUT_PATH),size=INPUT_PATH.stat().st_size,mtime_ns=INPUT_PATH.stat().st_mtime_ns),
        source_hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in
            [THIS_ROOT/'climatology_tools.py',THIS_ROOT/'direction_tools.py',THIS_ROOT/'02_shear_direction_preference.ipynb']})
    (OUTPUT/'settings.json').write_text(json.dumps(provenance,indent=2))
def table(frame, name, full=False):
    if SAVE_OUTPUTS: frame.to_csv(OUTPUT/f'{name}.csv',index=False)
    with pd.option_context('display.max_rows',None if full else 30,'display.max_columns',30): display(frame)
def show(fig, name):
    if SAVE_OUTPUTS: fig.savefig(OUTPUT/f'{name}.png',dpi=160,bbox_inches='tight')
    plt.show(); plt.close(fig)
def summary(frame, metrics, groups=('population','Cyc'), weighting='day'):
    return ct.cluster_summary(frame,metrics,groups=groups,weighting=weighting,n_boot=BOOTSTRAPS,
        min_eddies=MIN_EDDIES,min_rows=MIN_ROWS,seed=SEED)
def sector_plot(s, metric, name, reference=0):
    fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True,constrained_layout=True)
    for ax,population in zip(axes.flat,dt.POPULATIONS):
        for cyc in ('AE','CE'):
            q=s.loc[s.population.eq(population)&s.Cyc.eq(cyc)&s.metric.eq(metric)].set_index('sector').reindex(range(8))
            ax.errorbar(range(8),q['mean'].where(q.supported.eq(True)),
                yerr=np.vstack([q['mean']-q.ci_low,q.ci_high-q['mean']]).clip(0),
                fmt='o-',capsize=3,color=ct.COLORS[cyc],label=cyc)
        ax.axhline(reference,color='k',lw=.7)
        ax.set(title=population,xticks=range(8),xticklabels=dt.SECTORS,xlabel='Geographic shear direction',ylabel=ct.LABELS.get(metric, {'left_probability':'Fraction tilted left of shear'}.get(metric,metric)))
        ax.legend()
    show(fig,name)


## 1. Population support and geographic shear coverage

Shear is the upper-minus-lower velocity contrast (0–200 m minus 200–500 m), using the 91-day moving seasonal climatology. It is not the velocity direction itself. Bearings run clockwise from north. Compass sectors are centred on N, NE, E, etc.; N spans 337.5–22.5°.

A broad pooled shear distribution can arise from different eddies each experiencing steady shear. Section 4 separately measures variation **within** tracks over equal-duration windows. We do not assume topographic dominance implies a wider direction distribution.

In [ ]:
audit=data.assign(geometry_valid=np.isfinite(data.left_fraction)).groupby(['regime','Cyc']).agg(
    input_days=('Day','size'),geometry_valid_days=('geometry_valid','sum')).reset_index()
table(audit,'quality_audit',True)
support=pop.groupby(['population','Cyc']).agg(days=('Day','size'),eddies=('track_id','nunique'),
    median_lat=('lat','median'),median_shear=('shear_ms','median')).reset_index()
table(support,'population_support',True)
coverage=[]
fig,axes=plt.subplots(2,2,figsize=(12,7),sharey=True,constrained_layout=True)
for ax,population in zip(axes.flat,dt.POPULATIONS):
    for cyc in ('AE','CE'):
        q=pop.loc[pop.population.eq(population)&pop.Cyc.eq(cyc)]
        if q.empty: continue
        for weighting in ('day','eddy'):
            w=np.ones(len(q)) if weighting=='day' else 1/q.groupby(ct.KEYS).Day.transform('size').to_numpy()
            probs=np.bincount(q.sector,weights=w,minlength=8)/w.sum()
            resultant=abs(np.sum(w*np.exp(1j*np.deg2rad(q.shear_angle)))/w.sum())
            entropy=-sum(p*np.log(p) for p in probs if p>0)/np.log(8)
            coverage.append(dict(population=population,Cyc=cyc,weighting=weighting,
                circular_variance=1-resultant,sector_entropy=entropy,occupied_sectors=int((probs>0).sum())))
            if weighting=='eddy': ax.plot(range(8),100*probs,'o-',label=cyc,color=ct.COLORS[cyc])
    ax.set(title=population,xticks=range(8),xticklabels=dt.SECTORS,ylabel='Eddy-equal frequency (%)',xlabel='Shear direction')
    ax.legend()
table(pd.DataFrame(coverage),'pooled_shear_coverage',True)
show(fig,'geographic_shear_coverage')


## 2. Primary test: does the sign survive different shear directions?

The primary statistic is **mean sin(tilt angle − shear angle)**: positive is left and negative is right, without imposing an AE/CE sign. Also report the fraction on the left (0.5 reference), magnitude-weighted transverse displacement and along-shear component.

Whole-eddy bootstrap intervals account for repeated days within tracks. Primary weights are eddy-days; eddy-equal results are separate. Bins below 20 eddies or 50 days are gaps, not evidence for no effect. Confidence intervals are pointwise; do not count individually significant bins as a family-wide test.

The sector-balanced mean gives each of eight compass directions equal weight. It is only supported when **all eight** sectors pass sample thresholds. Its bootstrap resamples tracks jointly across sectors. This guards against a pooled result dominated by prevalent shear directions, but an overall mean alone cannot demonstrate consistency in every sector.

In [ ]:
metrics=['left_fraction','left_probability','left_km','parallel_fraction']
sector_results=pd.concat([summary(pop,metrics,('population','Cyc','sector'),w).assign(weighting=w)
                          for w in ('day','eddy')],ignore_index=True)
table(sector_results,'sector_statistics')
# Compact primary table is fully printed so saved notebook outputs remain reviewable.
compact=sector_results.loc[sector_results.metric.eq('left_fraction')&sector_results.weighting.eq('day')].copy()
compact['compass']=compact.sector.map(dict(enumerate(dt.SECTORS)))
table(compact[['population','Cyc','compass','mean','ci_low','ci_high','eddies','observations','supported']],
      'primary_sector_scorecard',True)
for w in ('day','eddy'):
    s=sector_results.loc[sector_results.weighting.eq(w)]
    sector_plot(s,'left_fraction',f'left_component_{w}')
sector_plot(sector_results.loc[sector_results.weighting.eq('day')],'left_probability','left_probability',.5)
balanced=dt.sector_balanced(pop,BOOTSTRAPS,MIN_EDDIES,MIN_ROWS,SEED)
table(balanced,'sector_balanced_scorecard',True)
# Geography check at finer resolution: export all latitude x sector estimates.
latpop=pop.assign(latitude_band=np.floor(pop.lat/2)*2+1)
lat_results=summary(latpop,['left_fraction'],('population','Cyc','latitude_band','sector'))
table(lat_results,'latitude_by_sector')
fig,axes=plt.subplots(2,2,figsize=(13,8),constrained_layout=True)
for i,population in enumerate(('planetary','topographic')):
    for j,cyc in enumerate(('AE','CE')):
        q=lat_results.loc[lat_results.population.eq(population)&lat_results.Cyc.eq(cyc)].copy()
        q['value']=q['mean'].where(q.supported)
        mat=q.pivot(index='latitude_band',columns='sector',values='value').reindex(columns=range(8))
        ax=axes[i,j]
        if len(mat):
            im=ax.imshow(np.ma.masked_invalid(mat.to_numpy()),origin='lower',aspect='auto',vmin=-1,vmax=1,cmap='RdBu_r')
            ax.set(yticks=range(len(mat)),yticklabels=mat.index)
            fig.colorbar(im,ax=ax,label='Mean left component')
        ax.set(title=f'{population} {cyc}',xticks=range(8),xticklabels=dt.SECTORS,xlabel='Shear direction',ylabel='Latitude band centre')
show(fig,'latitude_sector_control')


## 3. Does the geographic tilt direction move with the shear reference frame?

Conditional tilt-bearing distributions: each supported shear-sector column sums to 100%. A fixed geographic tilt preference would appear near a horizontal band. A shear-relative preference would move around the bearing circle as shear changes. The white reference marks show **exactly perpendicular** left (AE) or right (CE) as a visual reference, not a fitted or assumed tilt angle. The real preferred offset need not be 90°. Circular wrapping causes the reference to cross the plot boundary.

These are population distributions, not individual trajectories. Missing columns are unsupported shear sectors.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(12,9),constrained_layout=True)
for i,population in enumerate(('planetary','topographic')):
    for j,cyc in enumerate(('AE','CE')):
        q=pop.loc[pop.population.eq(population)&pop.Cyc.eq(cyc)].copy()
        q['tilt_bin']=np.floor((q.TiltDir%360)/15).astype(int)
        h=pd.crosstab(q.tilt_bin,q.sector).reindex(index=range(24),columns=range(8),fill_value=0).astype(float)
        h=h.div(h.sum(axis=0).replace(0,np.nan),axis=1)*100
        supported=compact.loc[compact.population.eq(population)&compact.Cyc.eq(cyc)].set_index('sector').supported
        for s in range(8):
            if not supported.get(s,False): h[s]=np.nan
        ax=axes[i,j]
        im=ax.imshow(np.ma.masked_invalid(h.to_numpy()),origin='lower',aspect='auto',extent=(-.5,7.5,0,360),cmap='viridis')
        target=(np.arange(8)*45+(-90 if cyc=='AE' else 90))%360
        ax.plot(range(8),target,'w_',ms=18,mew=2,label='Perpendicular reference')
        ax.set(title=f'{population} {cyc}',xticks=range(8),xticklabels=dt.SECTORS,
               yticks=[0,90,180,270,360],yticklabels=['N','E','S','W','N'],xlabel='Shear direction',ylabel='Tilt bearing')
        fig.colorbar(im,ax=ax,label='Conditional probability per 15° (%)')
show(fig,'tilt_bearing_given_shear')


## 4. Do topographic eddies experience more variable shear?

Use nonoverlapping **14 consecutive valid days** within a regime. Every window has equal duration, avoiding a larger apparent range simply because one group has longer tracks. Circular variance is 1 − |mean(exp(iθ))|: zero means constant direction, larger values mean less concentration. Mean absolute daily turn distinguishes gradual and rapidly changing shear. Net turn can be small even when the direction varies substantially.

Window means are bootstrapped by whole eddy; eddy-equal weights are also reported. Windows are not independent eddies. These are descriptive regime comparisons, not causal topography effects: regional composition, shear strength and survival into a 14-day window can differ. The all-population windows may cross regimes; regime-specific windows may not. Rejected days and gaps break every window.

In [ ]:
windows=dt.shear_windows(data,WINDOW_DAYS)
table(windows,'within_track_windows')
window_stats=pd.concat([summary(windows,['circular_variance','mean_abs_daily_turn','net_turn'],weighting=w)
                        for w in ('day','eddy')],ignore_index=True)
# In this table day weighting means equal window weight, not single-day observations.
window_stats['weighting']=window_stats.weighting.replace({'day':'window'})
table(window_stats,'within_track_variability',True)
fig,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
for ax,metric in zip(axes,['circular_variance','mean_abs_daily_turn']):
    for j,cyc in enumerate(('AE','CE')):
        q=window_stats.loc[window_stats.Cyc.eq(cyc)&window_stats.metric.eq(metric)&window_stats.weighting.eq('eddy')].set_index('population').reindex(dt.POPULATIONS)
        x=np.arange(4)+(j-.5)*.15
        ax.errorbar(x,q['mean'].where(q.supported.eq(True)),yerr=np.vstack([q['mean']-q.ci_low,q.ci_high-q['mean']]).clip(0),fmt='o',capsize=3,label=cyc,color=ct.COLORS[cyc])
    ax.set(xticks=range(4),xticklabels=dt.POPULATIONS,ylabel={'circular_variance':'Shear circular variance','mean_abs_daily_turn':'Mean absolute shear turn (degrees/day)'}[metric],title='Equal weight per eddy'); ax.legend()
show(fig,'within_track_shear_variability')


## 5. Within-track test: does the sign persist when shear turns?

Exact 3-, 7- and 14-day pairs, with **all intervening days valid** and in the same regime for regime-specific analyses. Bins describe the absolute shortest endpoint shear turn. They cannot resolve complete turns. Both positive and negative turns are pooled.

Report the raw left component at both endpoints, without conditioning on the initial tilt side. This avoids selecting only eddies that initially match the hypothesis. The joint fraction on the expected side at both endpoints is descriptive; **0.25 is not a valid independent-day null** because observations persist.

A second comparison evaluates the same final tilt against new shear versus frozen starting shear. Positive `new_minus_old_expected` means the hypothesised AE-left/CE-right projection is larger against the new shear. This is a geometric comparison, not a causal response coefficient. Conditioning on shear turns, spatial gradients, centre errors and shared environmental variation can affect it. Broad cross-sectional consistency alone cannot establish temporal adjustment.

In [ ]:
turn_tables=[]
for lag in TURN_LAGS:
    pairs=dt.turning_pairs(data,lag)
    s=summary(pairs,['start_left','end_left','end_left_old_shear','new_minus_old_expected','both_expected_side'],
              ('population','Cyc','turn_bin'))
    s['lag_days']=lag; turn_tables.append(s)
turn_results=pd.concat(turn_tables,ignore_index=True)
table(turn_results,'turning_shear_statistics')
primary=turn_results.loc[turn_results.lag_days.eq(PRIMARY_TURN_LAG)]
table(primary.loc[primary.metric.isin(['end_left','new_minus_old_expected'])], 'primary_turn_scorecard',True)
for metric in ['start_left','end_left','new_minus_old_expected']:
    fig,axes=plt.subplots(2,2,figsize=(12,7),sharey=True,constrained_layout=True)
    for ax,population in zip(axes.flat,dt.POPULATIONS):
        for cyc in ('AE','CE'):
            q=primary.loc[primary.population.eq(population)&primary.Cyc.eq(cyc)&primary.metric.eq(metric)].set_index('turn_bin').reindex(['0–15','15–45','45–90','90–180'])
            ax.errorbar(range(4),q['mean'].where(q.supported.eq(True)),yerr=np.vstack([q['mean']-q.ci_low,q.ci_high-q['mean']]).clip(0),fmt='o-',capsize=3,label=cyc,color=ct.COLORS[cyc])
        ax.axhline(0,color='k',lw=.7); ax.legend()
        ax.set(title=f'{population}: {PRIMARY_TURN_LAG} days',xticks=range(4),xticklabels=['0–15','15–45','45–90','90–180'],xlabel='Absolute shear turn (degrees)',ylabel={'start_left':'Initial left-of-shear component','end_left':'Final left-of-shear component','new_minus_old_expected':'Expected-side projection: new minus old shear'}[metric])
    show(fig,f'turning_shear_{metric}')


## 6. Prespecified sensitivity checks

Repeat the primary sector test with alternative shear and tilt thresholds, surface-to-deep shear, full-archive mean flow, stronger regime dominance, and shifted sector boundaries. Full-background shear asks a different timescale question; it is not an independent dataset. Equal eddy weighting was shown above.

Every sensitivity exports full sector estimates and plots, avoiding reliance on truncated tables. Inspect direction-specific disagreements and missing support rather than accepting a pooled mean. The summary counts supported bins and signs descriptively; these are not multiple-testing-corrected significance counts.

In [ ]:
if RUN_SENSITIVITIES:
    variants={'primary':{},'tilt_2km':{'min_tilt':2.},'tilt_10km':{'min_tilt':10.},
              'shear_001':{'min_shear':.01},'surface_deep':{'definition':'surface_deep'},
              'full_background':{'family':'full'},'dominance_3':{'factor':3.},'shifted_sectors':{}}
    sens=[]; checks=[]
    for name,options in variants.items():
        p=dt.populations(prepare(**options))
        if name=='shifted_sectors':
            p['sector']=np.floor(p.shear_bearing/45).astype(int)
        s=summary(p,['left_fraction'],('population','Cyc','sector')); s['variant']=name
        sens.append(s)
        # Shifted plot labels must describe the shifted centres accurately.
        if name=='shifted_sectors':
            old=dt.SECTORS
            dt.SECTORS=tuple(f'{22.5+45*i:g}°' for i in range(8))
        sector_plot(s,'left_fraction',f'sensitivity_{name}')
        if name=='shifted_sectors': dt.SECTORS=old
        for (population,cyc),q in s.groupby(['population','Cyc']):
            supported=q.loc[q.supported]; sign=1 if cyc=='AE' else -1
            checks.append(dict(variant=name,population=population,Cyc=cyc,supported_sectors=len(supported),
                expected_sign_sectors=int((sign*supported['mean']>0).sum()),
                opposite_sign_sectors=int((sign*supported['mean']<0).sum())))
    table(pd.concat(sens,ignore_index=True),'all_sensitivity_sectors')
    table(pd.DataFrame(checks),'sensitivity_support_and_signs',True)
print('Saved results:',OUTPUT if SAVE_OUTPUTS else 'saving disabled')


## Interpretation checklist

1. **Direction-relative organisation:** do supported compass sectors retain AE-left and CE-right, including latitude-stratified checks? Do geographic tilt distributions shift with shear bearing?
2. **Generality:** does the mixed population and all-quality population agree? Report missing sectors and unsupported groups. Never translate a mean preference into a rule obeyed by every eddy.
3. **Topographic hypothesis:** distinguish pooled geographic coverage from within-track variability. Report the result even if planetary shear is equally or more variable.
4. **Changing shear:** do endpoint signs survive substantial turns, and does new shear describe final tilt better than frozen old shear? Sparse large-turn bins are inconclusive.
5. **Limits:** these are associations with the seasonal background-flow contrast. Whole-column coherent tilt and layer-mean shear have different vertical supports. Eddy interactions and shared bathymetric/geographic controls are not eliminated by a large sample or whole-eddy bootstrap.

This notebook intentionally does **not** assign a formal p-value to a day-wise shuffle: that would destroy temporal dependence. The present cache lacks longitude and calendar metadata needed for a defensible location/season-matched surrogate. A suitable future null should preserve track autocorrelation and geographical/seasonal structure. Until then, supported results justify “tilt organised relative to shear,” not proof that shear alone causes tilt.